# DeepGuard — DF40 + DeepfakeBench + DeepGuard-LR

**One-click setup with diagnostics.** Run All is supported. The notebook stops safely at the first failed stage and prints a setup report. No external `lir` package is installed.


In [ ]:
import sys, subprocess, json, platform, os, traceback
from pathlib import Path
STATUS=[]
def stage(name, fn):
    print('\n'+'='*64); print(name); print('='*64)
    try:
        value=fn(); STATUS.append((name,'PASS',str(value))); print('PASS:',value); return value
    except Exception as e:
        STATUS.append((name,'FAIL',f'{type(e).__name__}: {e}')); print('FAIL:',type(e).__name__,e); traceback.print_exc(); return None


In [ ]:
def setup_drive():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    global ROOT
    ROOT=Path('/content/drive/MyDrive/DeepGuard')
    for p in ['datasets','models','features/xception','features/ftcn','features/fused','lir/development','lir/calibration','lir/validation','reports','logs','manifests']:
        (ROOT/p).mkdir(parents=True,exist_ok=True)
    return ROOT
stage('01 — Google Drive', setup_drive)


In [ ]:
def check_runtime():
    py=platform.python_version()
    gpu=subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null')
    return {'python':py,'gpu':gpu or 'NO GPU DETECTED'}
stage('02 — Runtime / GPU', check_runtime)


In [ ]:
def install_deepguard():
    import os
    if not Path('/content/deepguard-forensic-lr').exists():
        subprocess.run(['git','clone','https://github.com/geradts/deepguard-forensic-lr.git','/content/deepguard-forensic-lr'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','/content/deepguard-forensic-lr/requirements.txt'],check=True)
    return subprocess.getoutput('git -C /content/deepguard-forensic-lr rev-parse HEAD')
stage('03 — DeepGuard installation', install_deepguard)


In [ ]:
def check_python_stack():
    import numpy, scipy, sklearn, pandas, cv2, yaml, joblib
    import sys
    sys.path.insert(0,'/content/deepguard-forensic-lr')
    from deepguard_lir import LRFusionSystem, cllr, cllr_min
    return {'numpy':numpy.__version__,'scipy':scipy.__version__,'sklearn':sklearn.__version__,'pandas':pandas.__version__,'opencv':cv2.__version__,'joblib':joblib.__version__,'deepguard_lr':'OK'}
stage('04 — Python stack / DeepGuard-LR', check_python_stack)


In [ ]:
def test_lr():
    import numpy as np
    import sys; sys.path.insert(0,'/content/deepguard-forensic-lr')
    from deepguard_lir import LRFusionSystem, cllr, cllr_min
    rng=np.random.default_rng(7); X=np.r_[rng.normal(1,0.4,(40,2)),rng.normal(-1,0.4,(40,2))]; y=np.r_[np.ones(40,dtype=int),np.zeros(40,dtype=int)]
    m=LRFusionSystem().fit(X,y,['detector_a','detector_b']); llr=m.predict_llr(X); metrics=m.metrics(X,y)
    assert np.isfinite(llr).all() and metrics['cllr']>=0 and metrics['cllr_min']>=0
    return metrics
stage('05 — DeepGuard-LR self-test', test_lr)


In [ ]:
def clone_upstream():
    for name,url in [('DeepfakeBench','https://github.com/SCLBD/DeepfakeBench.git'),('DF40','https://github.com/YZY-stack/DF40.git')]:
        target=Path('/content')/name
        if not target.exists(): subprocess.run(['git','clone',url,str(target)],check=True)
    return 'DeepfakeBench and DF40 repositories ready'
stage('06 — Upstream repositories', clone_upstream)


In [ ]:
def save_environment():
    env={'python':platform.python_version(),'platform':platform.platform(),'git_deepguard':subprocess.getoutput('git -C /content/deepguard-forensic-lr rev-parse HEAD'),'gpu':subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null'),'status':STATUS}
    (ROOT/'logs/environment.json').write_text(json.dumps(env,indent=2))
    return ROOT/'logs/environment.json'
stage('07 — Audit manifest', save_environment)


In [ ]:
print('\n'+'#'*72); print('DEEPGUARD SETUP REPORT'); print('#'*72)
for name,status,detail in STATUS: print(f'{name:38} {status:5}  {detail}')
failed=[x for x in STATUS if x[1]=='FAIL']
print('\nOVERALL:', 'FAIL — fix the first failed stage above' if failed else 'PASS — environment is ready')


## After setup
If the final report says `PASS`, the next notebook cells can be added for dataset manifest creation and actual Xception/FTCN inference. The notebook intentionally does **not** download the full DF40 dataset or model checkpoints automatically.
